# EXP-011 — Zero-shot biểu cảm phức hợp RAF-DB (11 lớp)
Không train. Dùng checkpoint có sẵn, test trên RAF-DB compound (ảnh `aligned`, cùng loại với RAFDB basic). Mô hình chưa từng thấy 11 lớp này.

**Cách chấm điểm**
- *Prompt trực tiếp* (text mới cho lớp phức hợp): `Name` · `An expression` · `Photo` · `Compound Five` (5 câu dùng tính từ, VD "happily surprised") · `Conjunction Five` (đúng 5 template Ensemble Five, thay tên lớp bằng "happiness and surprise").
- *Tổ hợp từ 7 lớp cơ bản* (dùng lại prompt Ensemble Five của RAFDB, không thêm text mới): `Sum` = s_A + s_B · `Min` = min(s_A, s_B) · `EmbAvg` = cos(ảnh, chuẩn hoá(e_A + e_B)).

**Head:** CLIP gốc · Paper · EXP-000 · EXP-004 (anchor e30) · EXP-010 plain e5/e10 (3 seed).
**Tập:** `test` (792 ảnh, chính) và `all` = train+test (3954 ảnh, phụ). Tiền xử lý P0, thêm P0+flip.

**Tiêu chí (chọn trước):**
- Chính: với ít nhất một cách chấm, EXP-010 plain e5 (mean 3 seed, test, P0) có UAR ≥ CLIP gốc + 2.0 **và** hơn Paper (cùng cách chấm).
- Phụ: trên các head đã train, `Sum` ≥ prompt trực tiếp (`Conjunction Five`, `Compound Five`).

# Cấu hình

In [ ]:
SEEDS = [1, 2, 3]
BRANCH = "exp/010-affectnet-train"   # chỉ cần models/clip + models/Text.py (không cần lavis, không cần code mới)
COMPOUND_ZIP = "/content/drive/MyDrive/Datasets/RAFDB_compound.zip"   # từ tools/prepare_rafdb_compound.py
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"     # chỉ để kiểm pipeline trên RAFDB basic
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp011"
EXP000_ROOT = "/content/drive/MyDrive/ExpCLIP_output"  # tìm đệ quy expclip_l14_face_seed{N}-model.pth
EXP004_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
EXP010_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp010"
SPLITS = ["test", "all"]
CFG_NAMES = ["P0", "P0+flip"]
MAIN_SPLIT, MAIN_CFG = "test", "P0"
N_BOOT = 2000

# Mount Drive & GPU

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Clone repo & giải nén dữ liệu

In [ ]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
assert os.path.isdir("/content/Exp-CLIP/models"), "clone lỗi"
!cd /content/Exp-CLIP && git checkout $BRANCH && git pull && git log --oneline -1   # ghi lại commit vào research_log
assert os.path.exists(COMPOUND_ZIP), f"Chưa upload {COMPOUND_ZIP}"
!unzip -q -n "$COMPOUND_ZIP" -d /content
!ls /content/RAFDB_compound/test
!mkdir -p /content/static_test_sets
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
# CLIP vendored cần ftfy và pkg_resources (setuptools mới đã bỏ)
!pip install -q ftfy "setuptools<81"

# Tìm checkpoint
Thiếu file nào thì bỏ qua (in cảnh báo).

In [ ]:
import glob

def find_ckpt(root, name):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None

HEADS = {
    "CLIP gốc": {0: None},
    "Paper": {0: "/content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth"},
    "EXP-000": {s: find_ckpt(EXP000_ROOT, f"expclip_l14_face_seed{s}-model.pth") for s in SEEDS},
    "EXP-004": {s: f"{EXP004_DIR}/exp004_anchor_crossA_l1_l14_seed{s}-e30-model.pth" for s in SEEDS},
    "EXP-010 e5": {s: f"{EXP010_DIR}/exp010_affnet_plain_l14_seed{s}-e5-model.pth" for s in SEEDS},
    "EXP-010 e10": {s: f"{EXP010_DIR}/exp010_affnet_plain_l14_seed{s}-e10-model.pth" for s in SEEDS},
}
MAIN_HEAD = "EXP-010 e5"
for m, d in HEADS.items():
    for s, p in list(d.items()):
        if m != "CLIP gốc" and (p is None or not os.path.exists(p)):
            print(f"CẢNH BÁO: {m} seed {s}: không thấy checkpoint ({p}) → bỏ qua")
            del d[s]
        else:
            print(f"{m} seed {s}: {p}")
HEADS = {m: d for m, d in HEADS.items() if d}   # bỏ head không còn seed nào
assert MAIN_HEAD in HEADS and "CLIP gốc" in HEADS and "Paper" in HEADS, list(HEADS)

# Lớp & prompt
Thứ tự lớp = nhãn RAF-DB compound 1..11 (`compound/README.txt`), trùng thứ tự thư mục `00_…`–`10_…`.
Mỗi lớp = (cảm xúc trội A, cảm xúc phụ B) theo chỉ số 7 lớp cơ bản của RAFDB trong `models/Text.py`
(0 Happiness, 1 Sadness, 2 Neutral, 3 Anger, 4 Surprise, 5 Disgust, 6 Fear).

In [ ]:
import sys, json, numpy as np, pandas as pd, torch, torch.nn.functional as F
import torchvision.datasets as datasets, torchvision.transforms as T
sys.path.insert(0, "/content/Exp-CLIP")
from models.clip import clip
from models.Text import FER_prompt_list, FER_prompt_type_list, emotion_name_for_cm

BASIC = emotion_name_for_cm[:7]
assert BASIC == ["Happiness", "Sadness", "Neutral", "Anger", "Surprise", "Disgust", "Fear"], BASIC
H, S, N, A, SU, D, FE = range(7)
COMPOUND = [("Happily Surprised", H, SU), ("Happily Disgusted", H, D), ("Sadly Fearful", S, FE),
            ("Sadly Angry", S, A), ("Sadly Surprised", S, SU), ("Sadly Disgusted", S, D),
            ("Fearfully Angry", FE, A), ("Fearfully Surprised", FE, SU), ("Angrily Surprised", A, SU),
            ("Angrily Disgusted", A, D), ("Disgustedly Surprised", D, SU)]
CLS = [c[0] for c in COMPOUND]
PA = torch.tensor([c[1] for c in COMPOUND]); PB = torch.tensor([c[2] for c in COMPOUND])
NC = len(COMPOUND)
NOUN = [b.lower() for b in BASIC]
FIVE_TPL = ["an expression of {}.", "a photo of a face with an expression of {}.", "a photo radiating {} in a person.",
            "a photo showing someone immersed in {}.", "a photo capturing {} within an individual."]   # template của Ensemble Five

def direct_prompts(kind, name, a, b):
    n = name.lower()
    return {"Name": [f"{n}."],
            "An expression": [f"a {n} expression."],
            "Photo": [f"a photo of a face with a {n} expression."],
            "Compound Five": [f"a {n} expression.", f"a photo of a face with a {n} expression.",
                              f"a photo of a person who looks {n}.", f"a photo showing someone who is {n}.",
                              f"a photo capturing a {n} face."],
            "Conjunction Five": [t.format(f"{NOUN[a]} and {NOUN[b]}") for t in FIVE_TPL]}[kind]

DIRECT = ["Name", "An expression", "Photo", "Compound Five", "Conjunction Five"]
COMPOSE = ["Sum", "Min", "EmbAvg"]
METHODS = DIRECT + COMPOSE
DIRECT_TEXT = {k: [p for name, a, b in COMPOUND for p in direct_prompts(k, name, a, b)] for k in DIRECT}
BASIC_TEXT = dict(zip(FER_prompt_type_list, FER_prompt_list))["Expression Ensemble Five"]["RAFDB"]   # 35 câu, 7×5
for k in DIRECT:
    print(f"{k:17s} {DIRECT_TEXT[k][:len(DIRECT_TEXT[k]) // NC]}")
print("Basic (Sum/Min/EmbAvg):", BASIC_TEXT[:5])

# Feature CLIP L/14
Ảnh tính 1 lần (gốc + lật). `all` = ghép `test` + `train`.

In [ ]:
clip_model, _ = clip.load("ViT-L/14", "cuda")
clip_model.eval()
TF = T.Compose([T.Resize((224, 224)), T.ToTensor()])   # P0, giống test.py

def encode_folder(path):
    ds = datasets.ImageFolder(path, TF)
    loader = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=False, num_workers=4)
    f0, f1 = [], []
    with torch.no_grad():
        for x, _ in loader:
            x = x.cuda()
            f0.append(clip_model.encode_image(x).float().cpu())
            f1.append(clip_model.encode_image(x.flip(-1)).float().cpu())   # lật ngang
    return torch.cat(f0), torch.cat(f1), np.array(ds.targets), ds.classes

def encode_text(lst):
    with torch.no_grad():
        return clip_model.encode_text(clip.tokenize(lst, context_length=77, truncate=True).cuda()).float().cpu()

img, y = {}, {}
for split in ["test", "train"]:
    f0, f1, yy, folders = encode_folder(f"/content/RAFDB_compound/{split}")
    assert [c.split("_", 1)[1].replace("_", " ") for c in folders] == CLS, folders
    img[split], y[split] = (f0, f1), yy
    print(split, tuple(f0.shape), np.bincount(yy))
img["all"] = tuple(torch.cat([img["test"][k], img["train"][k]]) for k in range(2))
y["all"] = np.concatenate([y["test"], y["train"]])
del img["train"], y["train"]

txt = {k: encode_text(v) for k, v in DIRECT_TEXT.items()}
txt["basic"] = encode_text(BASIC_TEXT)

# RAFDB basic, chỉ để kiểm pipeline
rb0, _, rb_y, _ = encode_folder("/content/static_test_sets/RAFDB/test")

# Dự đoán & metric

In [ ]:
def load_W(path):
    if path is None:
        return None
    sd = torch.load(path, map_location="cpu")
    return next(v for k, v in sd.items() if "mlp.weight" in k).float()   # (4096, 768)

def proj(x, W):
    return F.normalize(x @ W.T if W is not None else x, dim=-1)

def basic_scores(i, W):
    return (i @ proj(txt["basic"], W).T).view(len(i), 7, -1).mean(-1)   # (N, 7), như test.py

def scores(W, feats, method):
    i = proj(feats, W)
    if method in DIRECT:
        return (i @ proj(txt[method], W).T).view(len(i), NC, -1).mean(-1)
    sb = basic_scores(i, W)
    if method == "Sum":
        return sb[:, PA] + sb[:, PB]
    if method == "Min":
        return torch.minimum(sb[:, PA], sb[:, PB])
    if method == "EmbAvg":
        tb = proj(txt["basic"], W)
        e = F.normalize(tb.view(7, -1, tb.shape[-1]).mean(1), dim=-1)   # 1 vector / lớp cơ bản
        return i @ F.normalize(e[PA] + e[PB], dim=-1).T

def uar(y_, p_, n=NC):
    cnt = np.bincount(y_, minlength=n)
    cor = np.bincount(y_[p_ == y_], minlength=n)
    return (cor[cnt > 0] / cnt[cnt > 0]).mean() * 100

PAIR = [{a, b} for _, a, b in COMPOUND]
def metrics(y_, p_):
    acc = np.array([(p_[y_ == c] == c).mean() * 100 for c in range(NC)])
    hit = np.array([len(PAIR[t] & PAIR[p]) > 0 for t, p in zip(y_, p_)])   # đoán đúng ít nhất 1 cảm xúc thành phần
    return {"UAR": acc.mean(), "WAR": (p_ == y_).mean() * 100, "comp_hit": hit.mean() * 100, "acc": acc}

rows, preds, basic_rows = [], {}, []
for m, d in HEADS.items():
    for s, path in d.items():
        W = load_W(path)
        for split in SPLITS:
            for cfg in CFG_NAMES:
                f0, f1 = img[split]
                feats = (f0 + f1) / 2 if cfg.endswith("+flip") else f0   # TTA: trung bình trước head
                # Lớp cơ bản mà mô hình đoán (7 lớp) có thuộc cặp đúng không?
                bp = basic_scores(proj(feats, W), W).argmax(1).numpy()
                in_pair = np.array([b in PAIR[t] for t, b in zip(y[split], bp)])
                dom = (bp == PA.numpy()[y[split]])
                basic_rows.append({"model": m, "seed": s, "split": split, "config": cfg, "basic_in_pair": in_pair.mean() * 100,
                                   "basic_is_dominant": dom.mean() * 100,
                                   **{f"basic_{BASIC[k]}": (bp == k).mean() * 100 for k in range(7)}})
                for meth in METHODS:
                    p = scores(W, feats, meth).argmax(1).numpy()
                    preds[(m, s, split, cfg, meth)] = p
                    r = metrics(y[split], p)
                    rows.append({"model": m, "seed": s, "split": split, "config": cfg, "method": meth,
                                 "UAR": r["UAR"], "WAR": r["WAR"], "comp_hit": r["comp_hit"],
                                 **{f"acc_{c}": a for c, a in zip(CLS, r["acc"])}})
res = pd.DataFrame(rows)
bas = pd.DataFrame(basic_rows)
res.to_csv(f"{OUT_DIR}/exp011_compound-results.csv", index=False)
bas.to_csv(f"{OUT_DIR}/exp011_compound-basic.csv", index=False)
print(len(res), "dòng →", OUT_DIR)

# Kiểm tra pipeline trên RAFDB basic (P0, Ensemble Five)
Phải khớp `test.py` đã ghi (lệch ≤ ~0.4 do fp16). Lệch lớn thì dừng lại.

In [ ]:
REF = {"EXP-000|1": [58.88, 66.88], "EXP-000|2": [57.82, 65.78], "EXP-000|3": [57.5, 63.3],
       "EXP-004|1": [58.73, 65.94], "EXP-004|2": [57.07, 64.77], "EXP-004|3": [58.26, 65.06]}
chk = []
for k, (u_, w_) in REF.items():
    m, s = k.split("|")
    if int(s) not in HEADS.get(m, {}):
        continue
    W = load_W(HEADS[m][int(s)])
    p = basic_scores(proj(rb0, W), W).argmax(1).numpy()
    chk.append({"key": k, "dUAR": uar(rb_y, p, 7) - u_, "dWAR": (p == rb_y).mean() * 100 - w_})
chk = pd.DataFrame(chk)
print(chk.round(2).to_string(index=False))
print(f"max |ΔUAR| = {chk.dUAR.abs().max():.2f} → {'OK' if chk.dUAR.abs().max() <= 0.5 else 'LỆCH — kiểm tra lại!'}")

# Bảng 1 — UAR 11 lớp theo cách chấm × head (test, P0; mean ± std qua seed)
Mức ngẫu nhiên: 9.09.

In [ ]:
pd.set_option("display.width", 250)
def fmt(x):
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

def table(val="UAR", split=MAIN_SPLIT, cfg=MAIN_CFG):
    r = res[(res.split == split) & (res.config == cfg)]
    t = r.groupby(["method", "model"])[val].agg(fmt).unstack("model").loc[METHODS, list(HEADS)]
    print(f"--- {val} | split={split} | {cfg}\n{t.to_string()}\n")

table("UAR")
table("WAR")

# Bảng 2 — Δ UAR so với CLIP gốc, bootstrap 95% CI (test, P0)
Bootstrap ảnh (cùng mẫu cho mọi head), UAR của head = trung bình 3 seed. Kiểm tiêu chí chính cho `EXP-010 e5`.

In [ ]:
yt = y[MAIN_SPLIT]
rng = np.random.default_rng(0)
boot_idx = [rng.integers(0, len(yt), len(yt)) for _ in range(N_BOOT)]

def head_uar(m, meth, idx=None):
    idx = np.arange(len(yt)) if idx is None else idx
    return np.mean([uar(yt[idx], preds[(m, s, MAIN_SPLIT, MAIN_CFG, meth)][idx]) for s in HEADS[m]])

ci_rows = []
for meth in METHODS:
    base = np.array([head_uar("CLIP gốc", meth, ix) for ix in boot_idx])
    for m in HEADS:
        if m == "CLIP gốc":
            continue
        b = np.array([head_uar(m, meth, ix) for ix in boot_idx]) - base
        ci_rows.append({"method": meth, "model": m, "dUAR": head_uar(m, meth) - head_uar("CLIP gốc", meth),
                        "lo": np.percentile(b, 2.5), "hi": np.percentile(b, 97.5)})
ci = pd.DataFrame(ci_rows)
ci["txt"] = ci.apply(lambda r: f"{r.dUAR:+.2f} [{r.lo:+.1f}, {r.hi:+.1f}]", axis=1)
print(ci.pivot(index="method", columns="model", values="txt").loc[METHODS].to_string(), "\n")

ok_any = []
for meth in METHODS:
    d_clip = ci[(ci.method == meth) & (ci.model == MAIN_HEAD)].dUAR.item()
    d_paper = head_uar(MAIN_HEAD, meth) - head_uar("Paper", meth)
    ok = d_clip >= 2.0 and d_paper > 0
    ok_any.append(ok)
    print(f"{meth:17s} {MAIN_HEAD} − CLIP gốc = {d_clip:+.2f} · − Paper = {d_paper:+.2f} → {'ĐẠT' if ok else '—'}")
print(f"\nTiêu chí chính: {'ĐẠT' if any(ok_any) else 'KHÔNG đạt'} (đạt ở {sum(ok_any)}/{len(METHODS)} cách chấm)")

# Bảng 3 — Tổ hợp vs prompt trực tiếp (Δ UAR theo cặp seed, test, P0)
Tiêu chí phụ: trên head đã train, `Sum` ≥ `Conjunction Five` và ≥ `Compound Five`.

In [ ]:
u = res[(res.split == MAIN_SPLIT) & (res.config == MAIN_CFG)].pivot_table(index=["model", "seed"], columns="method", values="UAR")
cmp_rows = []
for m in HEADS:
    x = u.loc[m]
    cmp_rows.append({"model": m,
                     "Sum − Conjunction Five": fmt(x["Sum"] - x["Conjunction Five"]),
                     "Sum − Compound Five": fmt(x["Sum"] - x["Compound Five"]),
                     "best compose − best direct": fmt(x[COMPOSE].max(1) - x[DIRECT].max(1))})
print(pd.DataFrame(cmp_rows).set_index("model").to_string())

# Bảng 4 — Đoán đúng thành phần
- `comp_hit`: lớp phức hợp đoán ra có chung ít nhất 1 cảm xúc với nhãn đúng (ngẫu nhiên ≈ tỉ lệ cặp chung thành phần).
- `basic_in_pair`: khi phân loại 7 lớp cơ bản, lớp đoán ra thuộc cặp đúng; `basic_is_dominant`: trùng cảm xúc trội.

In [ ]:
table("comp_hit")
b = bas[(bas.split == MAIN_SPLIT) & (bas.config == MAIN_CFG)]
t = b.groupby("model")[["basic_in_pair", "basic_is_dominant"] + [f"basic_{c}" for c in BASIC]].agg(fmt).loc[list(HEADS)]
print("--- Phân loại 7 lớp cơ bản trên ảnh compound (test, P0)\n" + t.to_string())
pair_rand = np.mean([[len(PAIR[t] & PAIR[p]) > 0 for p in range(NC)] for t in yt]) * 100
print(f"\ncomp_hit ngẫu nhiên (đoán đều 11 lớp) ≈ {pair_rand:.1f}")

# Bảng 5 — Accuracy từng lớp (test, P0, mean seed)

In [ ]:
r = res[(res.split == MAIN_SPLIT) & (res.config == MAIN_CFG)]
cols = [f"acc_{c}" for c in CLS]
for meth in ["Conjunction Five", "Compound Five", "Sum"]:
    t = r[r.method == meth].groupby("model")[cols].mean().loc[list(HEADS)]
    t.columns = [c.replace("ally ", ".").replace("ly ", ".") for c in CLS]
    print(f"--- {meth}  (số ảnh test: {np.bincount(yt).tolist()})\n{t.round(1).to_string()}\n")

# Bảng 6 — Tập `all` (3954 ảnh) và TTA lật

In [ ]:
table("UAR", split="all", cfg="P0")
table("UAR", split="test", cfg="P0+flip")

# Lưu JSON (cùng định dạng `research_log/results/`)

In [ ]:
out = {"source": "notebooks/EXP_011_compound_test.ipynb",
       "rows": [{"model": r.model, "seed": int(r.seed), "split": r.split, "config": r.config, "method": r.method,
                 "UAR": round(r.UAR, 2), "WAR": round(r.WAR, 2), "comp_hit": round(r.comp_hit, 2)} for r in res.itertuples()],
       "bootstrap_vs_clip": [{k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items() if k != "txt"}
                             for row in ci.to_dict("records")]}
with open(f"{OUT_DIR}/EXP-011-compound.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
print(f"{len(out['rows'])} dòng → {OUT_DIR}/EXP-011-compound.json")